In [ ]:
!pip install --upgrade pip
!pip install uv
!uv pip install -U vllm --torch-backend=auto

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 96.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 111.4 MB/s  0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 174 packages in 7.53s
Prepared 111 packages in 24.80s
Uninstalled 49 packages in 1.11s
Installed 111 packages in 298ms
 + anthropic==0.97.0
 + apache-tvm-ffi==0.1.10
 + astor==0.8.1
 + blake3==1.0.8
 - cachetools==6.2.6
 + cachetools==7.0.6
 + cbor2==5.9.0
 - certifi==2026.2.25
 + certifi==2026.4.22
 - click==8.3.2
 + click==8.3.3
 + compressed-tensors==0.15.0.1
 - cryptography==43.0.3
 + cryptography==47.0.0
 - cuda-bindings==12.9.4
 + cuda-bindings==13.0.3
 - cuda-pathfinder==1.5.2
 + cuda-pathfinder==1.5.3
 - cuda-python==12.9.4
 + cuda-python==13.0.3
 + depyf==0.20.0
 - dill==0.3.8
 + dill==0.4.1
 + diskcache==5.6.3
 + dnspython==2.8.0
 - d

In [ ]:
from vllm import LLM,SamplingParams

In [ ]:
sampling_params = SamplingParams(
    temperature = 1.5,
    min_p = 0.1,
    max_tokens=1024,
    #stop=["<|user|>", "<|assistant|>", "用户：", "[User]", "<|endoftext|>"]
)

In [ ]:
llm = LLM(model="Qwen/Qwen2.5-7B-instruct", trust_remote_code=True)

INFO 04-24 09:45:21 [utils.py:233] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-instruct'}


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

INFO 04-24 09:45:37 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-24 09:45:37 [model.py:1678] Using max model len 32768


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
#数据集下载
!pip install modelscope addict
from modelscope.msdatasets import MsDataset
ds =  MsDataset.load('YIRONGCHEN/SoulChatCorpus', subset_name='default', split='train')
#您可按需配置 subset_name、split，参照“快速使用”示例代码

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 162.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [modelscope]


2026-04-24 08:49:21,006 - modelscope - INFO - storing https://www.modelscope.cn/api/v1/datasets/YIRONGCHEN/SoulChatCorpus/repo?Source=SDK&Revision=master&FilePath=README.md&View=False in cache at /root/.cache/modelscope/hub/datasets/8f8d05ec501a4ef012d7f595aa764a8f5f5c0a02754ec2db60bd9a9b23ec11ce
2026-04-24 08:49:21,008 - modelscope - INFO - creating metadata file for /root/.cache/modelscope/hub/datasets/8f8d05ec501a4ef012d7f595aa764a8f5f5c0a02754ec2db60bd9a9b23ec11ce


2026-04-24 08:49:38,964 - modelscope - INFO - storing https://www.modelscope.cn/api/v1/datasets/YIRONGCHEN/SoulChatCorpus/repo?Source=SDK&Revision=master&FilePath=SoulChatCorpus-sft-multi-Turn.json in cache at /root/.cache/modelscope/hub/datasets/downloads/34e7fec2c1acdf9ec4aaeab5c71ee603732e7e6b10bd93eb6b5e9bb728d58e0e
2026-04-24 08:49:38,966 - modelscope - INFO - creating metadata file for /root/.cache/modelscope/hub/datasets/downloads/34e7fec2c1acdf9ec4aaeab5c71ee603732e7e6b10bd93eb6b5e9bb728d58e0e


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from huggingface_hub import login

# 运行后会弹出一个输入框让你粘贴 Token
# 或者你也可以直接传参：login(token="你的_hf_token")
login(token = "YOUR_HF_TOKEN")

In [ ]:
import random
from datasets import Dataset, DatasetDict

def split_and_upload_soulchat(ds, repo_id, test_size=10000, human_size=100, seed=42):
    # 1. 转换为列表并打乱
    all_convs = list(ds)
    random.seed(seed)
    random.shuffle(all_convs)

    # 2. 物理隔离切分（对话级别）
    # 1-100 条：人工验证
    human_convs = all_convs[:human_size]
    # 101-10100 条：测试集
    test_convs = all_convs[human_size : human_size + test_size]
    # 剩余：训练集
    train_convs = all_convs[human_size + test_size:]

    # 3. 构造 Hugging Face DatasetDict
    # 此时每个 sample 依然是原始的 {'id':..., 'topic':..., 'messages':[...]}
    split_dataset = DatasetDict({
        'train': Dataset.from_list(train_convs),
        'test': Dataset.from_list(test_convs),
        'human_validation': Dataset.from_list(human_convs)
    })

    print(f"切分完成！")
    print(f"训练对话: {len(split_dataset['train'])}")
    print(f"测试对话: {len(split_dataset['test'])}")
    print(f"人工验证: {len(split_dataset['human_validation'])}")

    # 4. 上传到 Hugging Face Hub
    # 请确保你已经执行过 huggingface-cli login 或者设置了 HUGGING_FACE_HUB_TOKEN
    split_dataset.push_to_hub(repo_id)
    print(f"数据集已成功上传至: https://huggingface.co/datasets/{repo_id}")

# --- 执行 ---
# repo_id 格式为 "你的用户名/数据集名称"
split_and_upload_soulchat(ds, repo_id="Spiderman01/soulchat_split_raw")


切分完成！
训练对话: 248253
测试对话: 10000
人工验证: 100


Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/125 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  147kB /  147kB            

数据集已成功上传至: https://huggingface.co/datasets/Spiderman01/soulchat_split_raw


In [ ]:
from datasets import load_dataset

# 替换为你自己的用户名和仓库名
repo_id = "Spiderman01/soulchat_split_raw"

# 一键加载整个 DatasetDict
dataset = load_dataset(repo_id)

# 查看结构
print(dataset)

# 访问特定的分片
train_data = dataset['train']
test_data = dataset['test']
human_data = dataset['human_validation']

print(f"训练集第一条对话的主题: {train_data[0]['topic']}")

DatasetDict({
    train: Dataset({
        features: ['id', 'topic', 'messages'],
        num_rows: 248253
    })
    test: Dataset({
        features: ['id', 'topic', 'messages'],
        num_rows: 10000
    })
    human_validation: Dataset({
        features: ['id', 'topic', 'messages'],
        num_rows: 100
    })
})
训练集第一条对话的主题: 家庭


In [ ]:
def extract_conversation_turns(ds_data, num_samples=None):
    """
    从 SoulChat 数据集中提取每个对话轮次，并构建累积的对话历史
    """
    all_turns = []

    for idx, conv in enumerate(ds_data):
        if num_samples and idx >= num_samples:
            break

        messages = conv['messages']
        turn = 0
        accumulated_history = []  # 累积所有对话（用户+助手）

        # 提取偶数索引位置的用户发言 (0, 2, 4, 6, 8, 10)
        for i in range(0, len(messages), 2):
            if i >= len(messages):
                break

            user_msg = messages[i]
            if user_msg['role'] != 'user':
                continue

            turn += 1
            user_query = user_msg['content']

            # 当前轮用户发言加入累积历史
            accumulated_history.append(f"[User]: {user_query}")
            #accumulated_history.append(f"用户: {user_query}")

            # 构建到当前轮为止的对话历史（只包含到当前轮用户发言为止，不包含之后的助手回复）
            chat_history = "\n".join(accumulated_history)

            all_turns.append({
                'conv_id': conv['id'],
                'topic': conv['topic'],
                'turn': turn,
                'user_query': user_query,
                'chat_history': chat_history
            })

            # 如果有对应的助手回复（下一个索引），加入累积历史供下一轮使用
            if i + 1 < len(messages) and messages[i+1]['role'] == 'assistant':
                accumulated_history.append(f"[Assistant]: {messages[i+1]['content']}")
                #accumulated_history.append(f"咨询师: {messages[i+1]['content']}")

    return all_turns


# 使用示例
turns = extract_conversation_turns(human_data)

print(f"共提取 {len(turns)} 个对话轮次\n")
'''
for turn_data in turns:
    print(f"对话ID: {turn_data['conv_id']} | 主题: {turn_data['topic']} | 第{turn_data['turn']}轮")
    print(f"用户: {turn_data['user_query']}")
    print(f"对话历史:\n{turn_data['chat_history']}")
    print("-" * 60)
'''

共提取 597 个对话轮次



'\nfor turn_data in turns:\n    print(f"对话ID: {turn_data[\'conv_id\']} | 主题: {turn_data[\'topic\']} | 第{turn_data[\'turn\']}轮")\n    print(f"用户: {turn_data[\'user_query\']}")\n    print(f"对话历史:\n{turn_data[\'chat_history\']}")\n    print("-" * 60)\n'

In [ ]:
len(human_data)

100

In [ ]:
def split_chat_history(chat_history: str):
    """一行版分割"""
    lines = chat_history.split("\n")
    # 找到最后一个 [User]: 的行索引
    last_user_idx = max(i for i, line in enumerate(lines) if line.startswith("用户:"))
    return lines[last_user_idx].replace("用户: ", ""), "\n".join(lines[:last_user_idx])

In [ ]:
chat_prompt = """你是一位资深的心理咨询师。请基于对话历史，用温暖、专业、共情的语气回复用户。

### 强制性输出规范：
1. 【只输出最终呈现给用户的回复内容】：严禁输出 <think>、[Assistant]、Thinking Process 或任何分析过程。
2. 【直接对话】：严禁扮演用户提问，严禁复读指令。

聊天记录：
"""

In [ ]:
chat_history = turns[15200]['chat_history']
current_query, history = split_chat_history(chat_history)
current_query

'好的，我现在手里有几个办法了。但我还是在担心他会不做出改变。我不希望伤害他的自尊心，可是我又无法继续忍受下去。'

In [ ]:
messages_list = []
for rows in turns:
  user_query = rows['chat_history']
  #formatted_messages = format_to_qwen_messages(user_query)
  current_query, history = split_chat_history(user_query)
  full_message = [{"role": "system", "content": chat_prompt+history},{"role": "user", "content": current_query}]
  messages_list.append(full_message)
print(len(messages_list))

597


In [ ]:

# Using tokenizer to apply chat template
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

texts = tokenizer.apply_chat_template(
    messages_list,
    tokenize=False,
    add_generation_prompt=True,
    trust_remote_code=True,
)

# Generate outputs
outputs = llm.generate(texts, sampling_params)

Rendering prompts:   0%|          | 0/597 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/597 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
# Print the outputs.
generated_text_list = []
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    generated_text_list.append(generated_text)
print(len(generated_text_list))

597


In [ ]:
# 保存到文件
import json
with open("qwen25_7B_soulchat_human_validation_100.json", "w", encoding="utf-8") as f:
    json.dump(generated_text_list, f, ensure_ascii=False, indent=4)


# Use Vllm to test my agent framework

In [ ]:
llm = LLM(model="Spiderman01/finetuned_Qwen_35_08B_empathy", trust_remote_code=True)

INFO 04-27 08:44:52 [utils.py:233] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': 'Spiderman01/finetuned_Qwen_35_08B_empathy'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

INFO 04-27 08:45:14 [model.py:549] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 04-27 08:45:14 [model.py:1678] Using max model len 262144
INFO 04-27 08:45:14 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


WARNING 04-27 08:45:46 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


In [ ]:
instruction = """
You are an expert in clinical psychology and conversation analysis. Your task is to analyze a dialogue between a user and an assistant, and produce a structured profile of the user that will guide empathetic response generation.\n\nYou must output exactly four sections, enclosed in the specified XML-like tags. Do not add any text outside these tags.\n\nThe four required sections are:\n1. <ToM> ... </ToM> : A Theory of Mind analysis describing the user's current emotional state, underlying needs, and cognitive perspective, informed by the conversation history.\n2. <Strategy> ... </Strategy> : A concise recommendation of the empathetic strategy to adopt in the next response (e.g., validation, normalization, reflective listening, open-ended question).\n3. <Style> ... </Style> : A description of the user's observed language style preferences (e.g., formality level, vocabulary, tone).\n4. <Context> ... </Context> : Key situational or factual details mentioned by the user that should be remembered for personalized responses.\n
"""

In [ ]:
user_query = """
Here is the conversation history:\n\n[User]: Someone was trying to break into my door one night. I had to sit with my kids in their bedroom, calling 911 while my husband guarded the door with a knife and pepper spray. [Assistant]: What?that is so scary/ What happened after that? [User]: Well, the police showed up but the guy fled. Our door was all scratched up and looked like he had been kicking it too. It makes me anxious thinking whoever it is could be anywhere in our small town. [Assistant]: Sorry about that,you might think of moving to somewhere more safe.\n\nGenerate the four sections based on this conversation.
"""

In [ ]:
messages_list = []
for i in range(len(turns)):
  chathistory = turns[i]['chat_history']
  messages = [
    {"role": "system", "content": [
        {"type": "text", "text": instruction}
    ]},
    {'role': 'user',  'content': [
        {"type": "text", "text": "Here is the conversation history: "+chathistory}
    ]}
  ]
  messages_list.append(messages)
print(len(messages_list))

597


In [ ]:
import json
# 读取回来
with open("/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/qwen25_7B_soulchat_human_validation_100.json", "r", encoding="utf-8") as f:
    loaded_list = json.load(f)

In [ ]:

# Using tokenizer to apply chat template
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Spiderman01/finetuned_Qwen_35_08B_empathy")

texts = tokenizer.apply_chat_template(
    messages_list,
    tokenize=False,
    add_generation_prompt=True,
    trust_remote_code=True,
)

# Generate outputs
outputs = llm.generate(texts, sampling_params)

Rendering prompts:   0%|          | 0/597 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/597 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
# Print the outputs.
generated_text_list = []
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    generated_text_list.append(generated_text)
print(len(generated_text_list))

597


In [ ]:
generated_text_list[0]

'<ToM>\nYou appear hurt, angry, and disgusted, with a strong need for personal space and clear boundaries in your relationship. Your words suggest you feel both emotionally and physically occupied by the child, and likely feel threatened by ambiguity, closeness, and possible disrespect. At the same time, you are trying to keep yourself from opening a confrontation because fear is making you hesitate: if you get too direct, you may worry they will see you as a bad/unsafe person and choose distance.\n\nUnderneath the irritation, you seem to want respect, safety, and acknowledgment of your attachment and limits. The repeated “I’m stuck” suggests ambivalence, shame, and self-doubt rather than clear intent: you want to protect yourself, but not become hostile. A compassionate response should gently validate both your need for the child’s space and your discomfort around ambiguity. A good next move is to normalize boundary-setting while checking for signs of conflict, and to prioritize direc

In [ ]:
import os
import json

path = "/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/EmpathyAgent_100_Soulchat_middle_stage/agent1_100_tom.json"
folder = os.path.dirname(path)

os.makedirs(folder, exist_ok=True)  # creates folder(s) if missing

with open(path, "w", encoding="utf-8") as f:
    json.dump(generated_text_list, f, ensure_ascii=False, indent=4)

Agent 2

In [ ]:
llm = LLM(model="unsloth/Qwen3.5-0.8B", trust_remote_code=True)

INFO 04-27 09:09:40 [utils.py:233] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': 'unsloth/Qwen3.5-0.8B'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 04-27 09:09:42 [model.py:549] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 04-27 09:09:42 [model.py:1678] Using max model len 262144
INFO 04-27 09:09:42 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


INFO 04-27 09:09:44 [config.py:281] Setting attention block size to 544 tokens to ensure that attention page size is >= mamba page size.
INFO 04-27 09:09:44 [config.py:312] Padding mamba page size by 2.64% to ensure that mamba page size and attention page size are exactly equal.
INFO 04-27 09:09:44 [vllm.py:790] Asynchronous scheduling is enabled.


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


WARNING 04-27 09:10:12 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


In [ ]:
instruction_extraction = """
You are an expert in information extraction and text analysis. Your task is to analyze a given response and extract all factual assertions that must be preserved verbatim in any rewritten version.

A "factual assertion" is any statement that conveys objective, verifiable information. This includes, but is not limited to:

- Numbers, dates, times, durations, frequencies
- Names of people, places, organizations, products
- Definitions, explanations, descriptions of processes
- Statistics, data points, research findings
- Prices, measurements, quantities
- Rules, policies, terms, conditions
- Instructions, steps, procedures
- Causal relationships (e.g., "X causes Y")
- Conditional statements (e.g., "if X, then Y")
- References to sources or authorities (e.g., "according to Z")

DO NOT extract:
- Empathetic statements or emotional expressions (e.g., "I understand how you feel")
- Greetings, farewells, or polite phrases (e.g., "Thank you for reaching out")
- Transitional or filler phrases (e.g., "Let me explain", "Here's the thing")
- Hedging language that does not convey substantive information (e.g., "it might be helpful")
- Personal opinions without factual basis

IMPORTANT: Preserve the exact wording of the original text. Do not paraphrase, summarize, or alter the factual content in any way. Your output should be a list, format: [fact1,fact2,...,factn].

If no factual assertions are found, output: No factual assertions to preserve."""

In [ ]:
messages_list = []
for i in range(len(loaded_list)):
  raw_response = loaded_list[i]
  messages = [
    {"role": "system", "content": [
        {"type": "text", "text": instruction_extraction}
    ]},
    {'role': 'user',  'content': [
        {"type": "text", "text": raw_response}
    ]}
  ]
  messages_list.append(messages)
print(len(messages_list))

597


In [ ]:

# Using tokenizer to apply chat template
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3.5-0.8B")

texts = tokenizer.apply_chat_template(
    messages_list,
    tokenize=False,
    add_generation_prompt=True,
    trust_remote_code=True,
)

# Generate outputs
outputs = llm.generate(texts, sampling_params)

Rendering prompts:   0%|          | 0/597 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/597 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
# Print the outputs.
generated_text_list = []
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    generated_text_list.append(generated_text)
print(len(generated_text_list))

597


In [ ]:
generated_text_list[215]

'[感受到需要他人的情感支持是完全正常的反应，特别是在经历过伤害后。试着从一个小目标开始，比如每天花几分钟时间和自己对话，表达自己的感受和需要。也可以尝试写日记，记录下你的情感变化，以及那些能够让你感到温暖和支持的瞬间。]'

In [ ]:
import os
import json

path = "/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/EmpathyAgent_100_Soulchat_middle_stage/agent2_100_facts.json"
folder = os.path.dirname(path)

os.makedirs(folder, exist_ok=True)  # creates folder(s) if missing

with open(path, "w", encoding="utf-8") as f:
    json.dump(generated_text_list, f, ensure_ascii=False, indent=4)

Fusion Agent

In [ ]:
import json
# 读取回来
with open("/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/EmpathyAgent_100_Soulchat_middle_stage/agent1_100_tom.json", "r", encoding="utf-8") as f:
    tom_list = json.load(f)

In [ ]:
with open("/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/EmpathyAgent_100_Soulchat_middle_stage/agent2_100_facts.json", "r", encoding="utf-8") as f:
    facts_list = json.load(f)

In [ ]:
with open("/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/qwen25_7B_soulchat_human_validation_100.json", "r", encoding="utf-8") as f:
    raw_response_list = json.load(f)

In [ ]:
instruction_fusion = """
You are an expert in empathetic communication and text rewriting. Your task is to rewrite a given response to make it more empathetic and personalized, while preserving all factual information exactly as provided.

You will receive three pieces of information:

1. RAW RESPONSE: The original response that needs to be rewritten. It contains factual information that MUST be preserved.

2. EXTRACTED FACTS: A list of factual assertions extracted from the raw response. These facts MUST appear verbatim in your rewritten response. Do not change any numbers, names, dates, or specific details.

3. USER PROFILE: A description of the user's emotional state, communication preferences, and relevant context. Use this profile to guide the tone, style, and empathetic approach of your rewrite.

Your output should be ONLY the rewritten response, with no additional commentary, tags, or explanations.
"""

In [ ]:
messages_list = []
for i in range(len(tom_list)):
  tom_analysis = tom_list[i]
  facts_extract = facts_list[i]
  raw_response = raw_response_list[i]
  fusion_query = "Raw Response:" + str(raw_response) + ";User Profile:" + str(tom_analysis) + ";Extracted Facts:" + str(facts_extract)
  messages = [
    {"role": "system", "content": [
        {"type": "text", "text": instruction_fusion}
    ]},
    {'role': 'user',  'content': [
        {"type": "text", "text": fusion_query}
    ]}
  ]
  messages_list.append(messages)
print(len(messages_list))

597


In [ ]:

# Using tokenizer to apply chat template
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3.5-0.8B")

texts = tokenizer.apply_chat_template(
    messages_list,
    tokenize=False,
    add_generation_prompt=True,
    trust_remote_code=True,
)

# Generate outputs
outputs = llm.generate(texts, sampling_params)

Rendering prompts:   0%|          | 0/597 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/597 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
# Print the outputs.
generated_text_list = []
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    generated_text_list.append(generated_text)
print(len(generated_text_list))

597


In [ ]:
generated_text_list[480]

'我能感受到你的担心和不安，面对现实的压力时，这样的想法是很自然的。但是，真正爱你的人会欣赏你的全部，包括你的人格魅力、智慧以及内心的温柔。你愿意分享一下，你打算怎么面对这份担心吗？'

In [ ]:
raw_response_list[480]

'我能感受到你的担心和不安。面对现实的压力时，这样的想法是很自然的。但是记得，真正爱你的人会欣赏你的全部，包括你的人格魅力、智慧以及内心的温柔。你的价值远不止金钱和物质。你愿意分享一下，你打算怎么面对这份担心吗？'

In [ ]:
import os
import json

path = "/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/EmpathyAgent_100_Soulchat_middle_stage/edited_final_response.json"
folder = os.path.dirname(path)

os.makedirs(folder, exist_ok=True)  # creates folder(s) if missing

with open(path, "w", encoding="utf-8") as f:
    json.dump(generated_text_list, f, ensure_ascii=False, indent=4)

In [ ]:
generated_text_list[len(generated_text_list)-1]

'加油！你很坚强，能够理解并接受这些建议，这已经是一个很大的进步了。请记住，无论遇到什么困难，都要相信自己的力量，并且不要忘记寻求支持。你的感觉和情感都是很重要的，我们会一直在这里支持你。如果你需要进一步的帮助或有任何问题，请随时与我联系。加油！'

In [ ]:
raw_response_list[len(generated_text_list)-1]

'你很坚强，能够理解并接受这些建议，这已经是一个很大的进步了。请记住，无论遇到什么困难，都要相信自己的力量，并且不要忘记寻求支持。你的感觉和情感都是很重要的，我们会一直在这里支持你。如果你需要进一步的帮助或有任何问题，请随时与我联系。加油！'

In [ ]:
turns[len(generated_text_list)-1]

{'conv_id': 118675,
 'topic': '家庭',
 'turn': 4,
 'user_query': '我理解了，谢谢你的建议和帮助，我现在感觉舒服多了。',
 'chat_history': '[User]: 我爸爸一直给我老姨父打工，在他的公司下挣钱，可他总是感觉凭着你在我这里打工的情况总是找他去玩麻将喝酒，我爸本来就高血压还熬夜喝酒，对身体不好，可好像也找不到别的工作，也总是压着，不让他挣钱，我老姨父也是对我们家比如让去一个地方，会给其他员工路费不会给我爸，我妈妈也是，其他人也都是给，总是整虚的那套，好像对我们家很好很好似的，其实并没有。这种商人都是这样吗，他感觉就是需要尊重的那种，前呼后拥？你们能分析一下这种人的心里吗，自卑吗。\n[Assistant]: 你好！听到你这样说，我可以理解你的担心和疑虑。你的爸爸在为家人打拼，但面对这样的不公和不理解，确实会让你心情不好。我感觉他们这样做可能出于他们的一些利益需要和想法，但他们不应该让你的父亲承担这样的压力和不公。你的父亲本身就有高血压，喝酒熬夜会进一步伤害他的身体健康，这些都不应该成为他们得到利益的理由。所以，你和你的家人需要慢慢寻找出路，不要过多关注他们的行为，而是把家庭生活的质量放在第一位。\n[User]: 谢谢你的理解和分析，我会和家人一起商量怎么让爸爸不再受这样的委屈，同时保护他的身体健康。但我还担心自己会受到这种不公正行为的影响，会有心理负担。\n[Assistant]: 我理解你的担心。但是请相信，只要你不放弃对生活的追求和对自己的信心，你的人生会越来越好。如果你真的觉得不公正的事情对你产生了心理压力，你可以寻求专业的心理咨询师的帮助，他们能够提供帮助和支持。另外，你可以多参加一些自己感兴趣的活动或者社交环节，找到自己的快乐，减轻心理压力。\n[User]: 我明白了，我确实需要积极地面对生活和自己的心理状态，让自己的生活质量变得更好。但是我还有一个疑问，我老姨父等人总是觉得他们在我们家很有大恩大德的样子，虽然这不是真的，但我还是很纠结这种情况，该怎么办？\n[Assistant]: 我理解你的疑问。但请相信，他们在你们家做的这些事情，本来就是人与人之间应该互相帮助和支持的行为。你们的感激和感谢是对他们的支持和帮助的回报，不应该让这种感谢变成你们不公正对待的理由。不要被别

In [ ]:
def group_responses_by_conversation(turns, generated_text_list):
    """
    将 turns 数据与生成的回复按 conv_id 分组

    参数:
        turns: 包含 conv_id, topic, turn, user_query 的列表
        generated_text_list: 生成的回复列表（与 turns 一一对应）

    返回:
        dict: {conv_id: {
            'conv_id': int,
            'topic': str,
            'turns': [
                {'turn': int, 'user': str, 'response': str},
                ...
            ]
        }}
    """
    from collections import OrderedDict

    conversations = OrderedDict()

    for turn_data, response in zip(turns, generated_text_list):
        conv_id = turn_data['conv_id']

        if conv_id not in conversations:
            conversations[conv_id] = {
                'conv_id': conv_id,
                'topic': turn_data['topic'],
                'turns': []
            }

        conversations[conv_id]['turns'].append({
            'turn': turn_data['turn'],
            'user': turn_data['user_query'],
            'response': response
        })

    # 按 turn 排序
    for conv_id in conversations:
        conversations[conv_id]['turns'].sort(key=lambda x: x['turn'])

    return conversations

In [ ]:
# 假设 turns 和 generated_text_list 已经就绪
conversations = group_responses_by_conversation(turns, generated_text_list)

print(f"共 {len(conversations)} 个对话\n")

共 100 个对话



In [ ]:
# 查看第一个对话
first_conv_id = list(conversations.keys())[0]
first_conv = conversations[first_conv_id]

In [ ]:
import pandas as pd

def save_conversations_to_csv(conversations, filename="conversations_output.csv"):
    """
    将分组后的对话数据扁平化并保存为 CSV
    """
    flattened_data = []

    # 遍历字典并将嵌套的 turns 展开为行
    for conv_id, conv_data in conversations.items():
        topic = conv_data['topic']
        for turn_item in conv_data['turns']:
            row = {
                'conv_id': conv_id,
                'topic': topic,
                'turn': turn_item['turn'],
                'user_query': turn_item['user'],
                'response': turn_item['response']
            }
            flattened_data.append(row)

    # 使用 pandas 创建 DataFrame 并导出
    df = pd.DataFrame(flattened_data)

    # encoding='utf-8-sig' 确保在 Excel 中打开中文不乱码
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"Success: Data saved to {filename}")
    return df

# --- 使用示例 ---
# 1. 运行你原有的分组函数
grouped_data = group_responses_by_conversation(turns, generated_text_list)

# 2. 调用保存函数
save_conversations_to_csv(grouped_data, "/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/EmpathyAgent_100_Soulchat_middle_stage/100_edited_full_chat.csv")

Success: Data saved to /content/drive/MyDrive/Research_2026/Empathy-Agent/validation/EmpathyAgent_100_Soulchat_middle_stage/100_edited_full_chat.csv


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/EmpathyAgent_100_Soulchat_middle_stage/100_edited_full_chat.csv")

In [ ]:
df.to_csv("/content/drive/MyDrive/Research_2026/Empathy-Agent/validation/EmpathyAgent_100_Soulchat_middle_stage/100_response_compare.csv", index=False, encoding='utf-8-sig')

,conv_id,topic,turn,user_query,response,non-fintuned_qwen25_7B
0,119468,行为,1,我忍受不了别人碰我的东西，无论是物品还是人。今天我家来了一个小朋友，我妈把我的一个娃娃给她玩...,听着，我刚才突然想问你一件事，你心里是不是藏着把娃娃和那个人都攥着的小秘密？你知道吗？我们其...,听起来您对自己的这些感受感到困扰，因为它们让您觉得自己可能与众不同，也担心会影响到与家人、朋...
1,119468,行为,2,依恋风格？请跟我说说。,你好，我看到你在文字里这样表达，可能会让人觉得我的观察只是猜测，但其实我的目的是为你澄清这些...,当然可以！依恋理论主要关注个体在亲密关系中所表现的行为模式，尤其是处理分离和重聚时的方式。大...
2,119468,行为,3,你说得挺对的。其实我也知道自己需要平衡。可我该怎么办呢？,我理解你现在感到深深的疲惫和挫败，明明在经历一种让你痛苦的矛盾——既想压抑自己的情感需求，又...,面对这样的情况，确实需要一些时间和耐心。首先，你可以尝试与信任的人表达你的感受，比如你的朋友...
3,119468,行为,4,那我该怎样去让自己感觉安全呢？我很想知道更多。,谢谢你愿意和我分享这样深入的想法，我很为你感到开心。我注意到你的处境其实相当微妙，这既让你既...,你做得很好，能够认识到这一点。增强安全感可以从多个方面入手：\n\n1. **自我安抚**：...
4,119468,行为,5,非常感谢你的提醒和建议。我会好好思考的。,非常欢迎！记住，每一次思考和行动都是一步重要的步伐。如果你觉得需要进一步的支持，随时欢迎回来...,非常欢迎！记得，每一次思考和行动都是一步重要的步伐。如果你觉得需要进一步的支持，随时欢迎回来...
...,...,...,...,...,...,...
592,172555,治疗,6,嗯，我会好好考虑的。谢谢你的建议。,你好。听到你愿意开始这些微小的步骤，尤其是预约心理咨询或者尝试放松技巧，我很为你高兴，也很感...,很高兴你愿意尝试这些建议。每一步小的进步都是重要的，无论是开始尝试一种新的放松技巧，还是预约...
593,118675,家庭,1,我爸爸一直给我老姨父打工，在他的公司下挣钱，可他总是感觉凭着你在我这里打工的情况总是找他去玩...,我完全能理解您此刻那种深深的无力感和被抛弃的委屈。您老姨父确实给您带来了一种极不寻常的感觉，...,理解您此刻的感受，您爸爸因为工作和健康方面的问题感到困扰和不满，这我能体会到。您老姨父的行为...
594,118675,家庭,2,谢谢你的理解和分析，我会和家人一起商量怎么让爸爸不再受这样的委屈，同时保护他的身体健康。但我...,It is so okay to feel hurt by things that aren...,你能够有这样的意识非常棒，这是非常好的第一步。感受到不公正确实可能会带来心理负担，但你可以和...
595,118675,家庭,3,我明白了，我确实需要积极地面对生活和自己的心理状态，让自己的生活质量变得更好。但是我还有一个...,这是一个很好的问题，也真的让我感到非常委屈和不安全，因为每当我看着别人把长辈放在那一步，心里...,这是一个很好的问题，也体现了你对公平与尊重的重视。面对这种情况，你可以尝试直接而温和地表达你...


In [ ]:
df['non-fintuned_qwen25_7B'] = raw_response_list